# Phase 2 TFIM-QRC v1.5: Full Topology + Virtual Nodes

This notebook tests a controlled upgrade of the best v1 static QRC branch, not the failed feedback branch.

Changes relative to v1 best:

- preserve PCA-6, 6 qubits, 6 anchors, ZXZZ, disorder=0.20, ridge alpha=3000;
- change Hamiltonian topology from chain to fully connected ZZ interactions;
- collect virtual-node observables after intermediate Trotter steps inside each anchor;
- sweep reservoir timescale via `evolution_time`.

Main question: does richer TFIM mixing + virtual-node temporal multiplexing improve reservoir diagnostics and out-of-sample performance without changing the whole modeling framework?

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.qrc.tfim_reservoir import (
    TFIMQRCConfig,
    diagnose_reservoir_feature_splits,
    fit_tfim_qrc_regressor,
    make_qrc_sequence_splits,
    summarize_qrc_result,
)

## 1. Data and PCA-6 sequence windows

In [ ]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

sequence_splits_6 = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

display(pca6.explained_variance)
print({name: (X.shape, y.shape) for name, (X, y, dates) in sequence_splits_6.items()})

## 2. v1.5 timescale sweep

Fixed architecture:

```text
6q / PCA-6 / 6 anchors / ZXZZ
topology = full
trotter_steps_per_anchor = 3
virtual_nodes_per_anchor = 3
disorder_strength = 0.20
ridge_alpha = 3000
```

Sweep only `evolution_time`.

In [ ]:
v15_rows = []
v15_diag_rows = []
v15_results = {}

for evolution_time in [0.5, 1.5, 3.0, 5.0]:
    config = TFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        anchor_count=6,
        anchor_policy="even",
        observable_mode="zxzz",
        collect_anchor_features=True,
        topology="full",
        trotter_steps_per_anchor=3,
        virtual_nodes_per_anchor=3,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=float(evolution_time),
        angle_max=3.141592653589793 / 2,
        ridge_alpha=3000.0,
        target_transform="log",
        seed=42,
        use_disorder=True,
        disorder_strength=0.20,
    )

    run_name = f"v15_full_virtualnodes_evolution_{evolution_time}"
    print(f"Running {run_name}")

    result = fit_tfim_qrc_regressor(
        sequence_splits_6,
        config=config,
        target=target,
        verbose=True,
    )

    v15_results[run_name] = result

    row = summarize_qrc_result(result)
    row["run_name"] = run_name
    v15_rows.append(row)

    _, y_train, _ = sequence_splits_6["train"]
    _, y_val, _ = sequence_splits_6["val"]
    _, y_test, _ = sequence_splits_6["test"]

    diag = diagnose_reservoir_feature_splits(
        result.train_features,
        result.val_features,
        result.test_features,
        y_train,
        y_val,
        y_test,
    )
    diag.insert(0, "run_name", run_name)
    v15_diag_rows.append(diag)

v15_table = pd.DataFrame(v15_rows)
v15_diagnostics = pd.concat(v15_diag_rows, ignore_index=True)

## 3. Metrics

In [ ]:
metric_cols = [
    "run_name",
    "topology",
    "trotter_steps_per_anchor",
    "virtual_nodes_per_anchor",
    "evolution_time",
    "n_reservoir_features",
    "train_rmse",
    "val_rmse",
    "test_rmse",
    "train_qlike",
    "val_qlike",
    "test_qlike",
    "train_mz_r2",
    "val_mz_r2",
    "test_mz_r2",
]

v15_table[metric_cols].sort_values("test_rmse")

## 4. Diagnostics

In [ ]:
diagnostic_cols = [
    "run_name",
    "split",
    "n_samples",
    "n_features",
    "near_constant_features",
    "feature_std_min",
    "feature_std_median",
    "feature_std_max",
    "effective_rank",
    "condition_number",
    "mean_abs_feature_target_corr",
    "max_abs_feature_target_corr",
    "mean_abs_shift_vs_train",
    "max_abs_shift_vs_train",
]

v15_diagnostics[diagnostic_cols]

## 5. Save outputs

In [ ]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

v15_table.to_csv(out_dir / "phase2_tfim_qrc_v15_full_virtualnodes_probe.csv", index=False)
v15_diagnostics.to_csv(out_dir / "phase2_tfim_qrc_v15_full_virtualnodes_diagnostics.csv", index=False)

print("Saved v1.5 full-topology virtual-node outputs to", out_dir)

## 6. Interpretation rule

Compare against current best v1:

```text
6q / PCA-6 / ZXZZ / snapshots / chain / disorder=0.20 / alpha=3000
test RMSE  = 0.102618
test QLIKE = -1.942716
test MZ R² = 0.072134
```

This test should be judged first by reservoir diagnostics, then by validation/test metrics. If all evolution times degrade feature stability and test performance, stop this architecture branch. If one timescale improves diagnostics and validation performance, run a smaller second sweep around that timescale.